# JurisEC — Notebook 1: Baseline Evaluation

**Goal.** Reproduce Table 6 of the paper: the overall retrieval performance of BM25 and three multilingual dense models (BGE-M3, multilingual-E5-large, and EmbeddingGemma-300M) on JurisEC.

**What it does.** Every query is ranked against the whole candidate collection (5,536 queries and 5,508 documents). Each system is evaluated in its own section, with its settings and prompts written out explicitly in the code.

**Sections.**
1. Data and relevance judgments
2. Evaluation functions
3. BM25
4. BGE-M3
5. multilingual-E5-large
6. EmbeddingGemma-300M
7. Results table (Table 6)

**Metrics.** nDCG@10, MRR@10, Acc@1, Acc@5, Acc@10, R@100, and MAP@100. Dense models are scored with `InformationRetrievalEvaluator` (sentence-transformers) and BM25 with `pytrec_eval`.

**Requirements.** A GPU and the `HF_TOKEN` secret in Colab (EmbeddingGemma is a gated model).

**Outputs.** `results/overall_results.csv` (the results table), `results/qrels.json` (the relevance judgments), and the embeddings in `cache_emb/`, which the other notebooks reuse.

**Reproducibility note.** The dense models run in half precision, so scores may vary in the third decimal place depending on the hardware.


In [1]:
%%capture
!pip install -q -U sentence-transformers datasets rank_bm25 pytrec_eval-terrier


In [2]:
import os, re, json, unicodedata
import numpy as np, pandas as pd, torch
from datasets import load_dataset

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = "/content/cache_emb"      # shared with the other notebooks
OUT   = "/content/results"
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)

RESULTADOS = {}    # each section stores its metrics here: RESULTADOS["BGE-M3"] = {...}
print("Device:", DEVICE)


Device: cuda


## 1. Data and relevance judgments

Loads the queries and their official responses. Identical responses are collapsed into a single candidate document. Nearly identical queries (TF-IDF cosine similarity of 0.90 or higher) receive reciprocal relevance judgments: the response to one also counts as relevant for the other.

The printed counts should match the paper: 5,536 queries, 5,508 unique documents, and 29 pairs of nearly identical queries.


In [3]:
ds = load_dataset("wilfredomartel/JurisEC")
ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
df = ds.to_pandas()

def limpia_texto(x):
    '''Strips surrounding whitespace; returns an empty string if the value is not text.'''
    return x.strip() if isinstance(x, str) else ""

for col in ["consulta", "resolucion_consulta", "materia", "fuente", "tema"]:
    if col in df.columns: df[col] = df[col].map(limpia_texto)
df = df[(df["consulta"] != "") & (df["resolucion_consulta"] != "")].reset_index(drop=True)

# Corpus: collapse identical responses into a single document
texto_a_docid, corpus_ids, corpus_texts = {}, [], []
for t in df["resolucion_consulta"]:
    if t not in texto_a_docid:
        texto_a_docid[t] = f"d{len(corpus_texts)}"
        corpus_ids.append(texto_a_docid[t]); corpus_texts.append(t)
df["doc_id"] = df["resolucion_consulta"].map(texto_a_docid)
df["qid"]    = [f"q{i}" for i in range(len(df))]
qids, queries_texts = df["qid"].tolist(), df["consulta"].tolist()
print(f"Queries: {len(df):,} | Unique documents: {len(corpus_texts):,} | Collapsed duplicates: {len(df)-len(corpus_texts)}")


README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/evaluation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.2MB            

data/evaluation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating evaluation split:   0%|          | 0/5536 [00:00<?, ? examples/s]

Queries: 5,536 | Unique documents: 5,508 | Collapsed duplicates: 28


In [4]:
# Nearly identical queries (TF-IDF cosine similarity >= 0.90): their responses are relevant to each other
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def normaliza(t):
    '''Lowercases the text and removes accents.'''
    t = unicodedata.normalize("NFD", t.lower())
    return "".join(c for c in t if unicodedata.category(c) != "Mn")

vectorizador = TfidfVectorizer(preprocessor=normaliza, token_pattern=r"[a-z0-9]{3,}")
matriz_tfidf = vectorizador.fit_transform(queries_texts)
pares_gemelas = []
for inicio in range(0, matriz_tfidf.shape[0], 512):
    sim = np.asarray(cosine_similarity(matriz_tfidf[inicio:inicio+512], matriz_tfidf))
    for fila in range(sim.shape[0]):
        i = inicio + fila; sim[fila, i] = -1.0
        j = int(sim[fila].argmax())
        if sim[fila, j] >= 0.90 and i < j:
            pares_gemelas.append((i, j))
print(f"Nearly identical query pairs: {len(pares_gemelas)}")

# qrels: each query -> its set of relevant documents
qrels = {q: {} for q in qids}
for q, d in zip(qids, df["doc_id"]): qrels[q][d] = 1
for i, j in pares_gemelas:
    qrels[qids[i]][df["doc_id"].iloc[j]] = 1
    qrels[qids[j]][df["doc_id"].iloc[i]] = 1
json.dump(qrels, open(f"{OUT}/qrels.json", "w"))
print(f"Queries with more than one relevant document: {sum(1 for q in qrels if len(qrels[q]) > 1)}")


Nearly identical query pairs: 29
Queries with more than one relevant document: 58


## 2. Evaluation functions

Two functions that report the same metrics, so that all systems are comparable:

- `evaluar_denso` scores a dense model with `InformationRetrievalEvaluator`, using the embeddings already computed in that model's section. The embeddings are passed to the evaluator as they are, without encoding the texts again.
- `evaluar_bm25` computes the same metrics for BM25 with `pytrec_eval`.

Both print the metrics and store them in `RESULTADOS` for the final table.


In [5]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
import pytrec_eval

class _FichaSeñuelo:
    '''The evaluator writes metrics to the model card; here they are received and discarded.'''
    def set_evaluation_metrics(self, *args, **kwargs): pass

class ServidorDeEmbeddings:
    '''Gives the evaluator the embeddings already computed (same text -> same row).
    The evaluator's ranking and metric logic runs unchanged.'''
    def __init__(self, textos_consultas, emb_consultas, textos_corpus, emb_corpus):
        self.ubicacion = {t: ("consulta", i) for i, t in enumerate(textos_consultas)}
        self.ubicacion.update({t: ("documento", i) for i, t in enumerate(textos_corpus)})
        self.emb_consultas, self.emb_corpus = emb_consultas, emb_corpus
        self.model_card_data = _FichaSeñuelo()
    def encode(self, sentences, **kw):
        filas = []
        for texto in sentences:
            origen, fila = self.ubicacion[texto]
            filas.append((self.emb_consultas if origen == "consulta" else self.emb_corpus)[fila])
        M = np.stack(filas)
        return torch.tensor(M) if kw.get("convert_to_tensor") else M
    encode_query = encode_document = encode

def evaluar_denso(nombre, textos_consultas, emb_consultas, textos_corpus, emb_corpus):
    '''Scores a dense model with InformationRetrievalEvaluator and stores its metrics.'''
    evaluador = InformationRetrievalEvaluator(
        queries=dict(zip(qids, textos_consultas)),
        corpus=dict(zip(corpus_ids, textos_corpus)),
        relevant_docs={q: set(qrels[q]) for q in qids},
        name=f"jurisec_{nombre}",
        ndcg_at_k=[10], mrr_at_k=[10], accuracy_at_k=[1, 5, 10],
        precision_recall_at_k=[5, 10, 100], map_at_k=[100],
        score_functions={"cos_sim": cos_sim}, show_progress_bar=False)
    servidor = ServidorDeEmbeddings(textos_consultas, emb_consultas, textos_corpus, emb_corpus)
    crudo = evaluador(servidor)
    def valor(sufijo):
        return next(v for k, v in crudo.items() if k.endswith(sufijo))
    RESULTADOS[nombre] = {
        "nDCG@10": valor("ndcg@10"), "MRR@10": valor("mrr@10"),
        "Acc@1": valor("accuracy@1"), "Acc@5": valor("accuracy@5"), "Acc@10": valor("accuracy@10"),
        "R@100": valor("recall@100"), "MAP@100": valor("map@100")}
    print(f"--- {nombre} ---")
    for k, v in RESULTADOS[nombre].items(): print(f"  {k}: {v:.4f}")
    return RESULTADOS[nombre]

def evaluar_bm25(nombre, run):
    '''Scores a run (dict query -> {doc_id: score}) with pytrec_eval; same metrics.'''
    evaluador = pytrec_eval.RelevanceEvaluator(
        qrels, {"ndcg_cut_10", "recip_rank", "recall_100", "map_cut_100"})
    por_consulta = evaluador.evaluate(run)
    def top_ordenado(q, k):
        return [d for d, _ in sorted(run[q].items(), key=lambda kv: -kv[1])[:k]]
    acc = {k: np.mean([any(d in qrels[q] for d in top_ordenado(q, k)) for q in qids]) for k in (1, 5, 10)}
    mrr10 = []
    for q in qids:
        pos = next((p for p, d in enumerate(top_ordenado(q, 10), 1) if d in qrels[q]), None)
        mrr10.append(1.0 / pos if pos else 0.0)
    RESULTADOS[nombre] = {
        "nDCG@10": np.mean([m["ndcg_cut_10"] for m in por_consulta.values()]),
        "MRR@10": float(np.mean(mrr10)),
        "Acc@1": float(acc[1]), "Acc@5": float(acc[5]), "Acc@10": float(acc[10]),
        "R@100": np.mean([m["recall_100"] for m in por_consulta.values()]),
        "MAP@100": np.mean([m["map_cut_100"] for m in por_consulta.values()])}
    print(f"--- {nombre} ---")
    for k, v in RESULTADOS[nombre].items(): print(f"  {k}: {v:.4f}")
    return RESULTADOS[nombre]


/tmp/ipykernel_6048/3017650808.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


## 3. BM25 (lexical baseline)

Ranks documents by the words they share with the query, using Okapi BM25 with the library defaults ($k_1 = 1.5$, $b = 0.75$). Text is lowercased and accents are removed; words shorter than three characters and Spanish stop words are dropped.


In [6]:
from rank_bm25 import BM25Okapi

STOPWORDS = set(("de la el en y a los del las un una que por con para es al se su o e lo como mas "
"pero sus le ya fue este si porque esta entre cuando muy sin sobre tambien me hasta hay donde quien "
"desde todo nos durante todos uno les ni contra otros ese eso ante ellos esto mi antes algunos unos "
"yo otro otras otra tanto esa estos mucho quienes nada muchos cual sea poco ella estar haber estas "
"estaba estamos algunas algo nosotros").split())

def tokenize(texto):
    '''Words of 3+ characters, lowercased, without accents or stop words.'''
    return [w for w in re.findall(r"[a-z0-9]+", normaliza(texto)) if len(w) > 2 and w not in STOPWORDS]

indice_bm25 = BM25Okapi([tokenize(d) for d in corpus_texts])

run_bm25 = {}
for i, q in enumerate(qids):
    puntajes = indice_bm25.get_scores(tokenize(queries_texts[i]))   # scored against the WHOLE corpus
    top = np.argsort(-puntajes)[:1000]                              # the top 1,000 are kept (TREC convention)
    run_bm25[q] = {corpus_ids[j]: float(puntajes[j]) for j in top}

evaluar_bm25("BM25", run_bm25)


--- BM25 ---
  nDCG@10: 0.7672
  MRR@10: 0.7338
  Acc@1: 0.6568
  Acc@5: 0.8326
  Acc@10: 0.8736
  R@100: 0.9465
  MAP@100: 0.7359


{'nDCG@10': np.float64(0.7672059884860105),
 'MRR@10': 0.7337960334663731,
 'Acc@1': 0.6567919075144508,
 'Acc@5': 0.8325505780346821,
 'Acc@10': 0.8735549132947977,
 'R@100': np.float64(0.9465317919075145),
 'MAP@100': np.float64(0.7359135128479779)}

## 4. BGE-M3 (dense, 8,192 tokens)

Its model card defines no prefixes, so queries and documents are encoded as they are.


In [7]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("BAAI/bge-m3", device=DEVICE,
                             model_kwargs={"torch_dtype": torch.float16})
modelo.max_seq_length = 8192

emb_consultas = modelo.encode(queries_texts, batch_size=8, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
emb_corpus    = modelo.encode(corpus_texts, batch_size=8, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)

np.save(f"{CACHE}/BGE-M3_Q.npy", emb_consultas)   # reused by the other notebooks
np.save(f"{CACHE}/BGE-M3_D.npy", emb_corpus)

evaluar_denso("BGE-M3", queries_texts, emb_consultas, corpus_texts, emb_corpus)
del modelo; torch.cuda.empty_cache()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (10 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/692 [00:00<?, ?it/s]

Batches:   0%|          | 0/689 [00:00<?, ?it/s]

--- BGE-M3 ---
  nDCG@10: 0.7742
  MRR@10: 0.7413
  Acc@1: 0.6671
  Acc@5: 0.8398
  Acc@10: 0.8786
  R@100: 0.9486
  MAP@100: 0.7436


## 5. multilingual-E5-large (dense, 512 tokens)

Its model card requires the prefixes `query: ` and `passage: `. Documents longer than 512 tokens are truncated.


In [8]:
modelo = SentenceTransformer("intfloat/multilingual-e5-large", device=DEVICE,
                             model_kwargs={"torch_dtype": torch.float16})
modelo.max_seq_length = 512

consultas_con_prefijo = ["query: " + q for q in queries_texts]      # official prefix, written out
corpus_con_prefijo    = ["passage: " + d for d in corpus_texts]

emb_consultas = modelo.encode(consultas_con_prefijo, batch_size=32, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
emb_corpus    = modelo.encode(corpus_con_prefijo, batch_size=32, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)

np.save(f"{CACHE}/E5_Q.npy", emb_consultas)
np.save(f"{CACHE}/E5_D.npy", emb_corpus)

evaluar_denso("E5", consultas_con_prefijo, emb_consultas, corpus_con_prefijo, emb_corpus)
del modelo; torch.cuda.empty_cache()


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/173 [00:00<?, ?it/s]

Batches:   0%|          | 0/173 [00:00<?, ?it/s]

--- E5 ---
  nDCG@10: 0.7527
  MRR@10: 0.7200
  Acc@1: 0.6481
  Acc@5: 0.8165
  Acc@10: 0.8573
  R@100: 0.9305
  MAP@100: 0.7222


## 6. EmbeddingGemma-300M (dense, 2,048 tokens)

A gated model: it needs the `HF_TOKEN` secret in Colab. Its model card defines instruction prompts that are placed before each text. The prompts are written out in the code and checked against the ones stored in the model. The retrieval prompt (`search result`) is used. The model runs in bfloat16 when the GPU supports it.


In [9]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()   # interactive login if no secret is set

dtype_gemma = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
modelo = SentenceTransformer("google/embeddinggemma-300m", device=DEVICE,
                             model_kwargs={"torch_dtype": dtype_gemma})
modelo.max_seq_length = 2048

PROMPT_CONSULTA  = "task: search result | query: "     # from the official model card
PROMPT_DOCUMENTO = "title: none | text: "
assert modelo.prompts.get("query") == PROMPT_CONSULTA, modelo.prompts
assert modelo.prompts.get("document") == PROMPT_DOCUMENTO, modelo.prompts

consultas_con_prompt = [PROMPT_CONSULTA + q for q in queries_texts]
corpus_con_prompt    = [PROMPT_DOCUMENTO + d for d in corpus_texts]

emb_consultas = modelo.encode(consultas_con_prompt, batch_size=16, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
emb_corpus    = modelo.encode(corpus_con_prompt, batch_size=16, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)

np.save(f"{CACHE}/GEMMA_Q.npy", emb_consultas)
np.save(f"{CACHE}/GEMMA_D.npy", emb_corpus)

evaluar_denso("GEMMA", consultas_con_prompt, emb_consultas, corpus_con_prompt, emb_corpus)
del modelo; torch.cuda.empty_cache()


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

... (13 outputs left out)

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

Batches:   0%|          | 0/346 [00:00<?, ?it/s]

Batches:   0%|          | 0/345 [00:00<?, ?it/s]

--- GEMMA ---
  nDCG@10: 0.7815
  MRR@10: 0.7507
  Acc@1: 0.6810
  Acc@5: 0.8409
  Acc@10: 0.8790
  R@100: 0.9503
  MAP@100: 0.7532


## 7. Results table (Table 6)

Collects the metrics stored by each section into one table, reports the best system for each metric, and saves the table as a CSV file. Significance tests for these differences are in Notebook 4.


In [10]:
tabla = pd.DataFrame(RESULTADOS).T.round(4)
tabla["N_top1"] = (tabla["Acc@1"] * len(qids)).round().astype(int)   # queries answered correctly at rank one
print(tabla.to_string())
print("\nBest system per metric:")
for col in ["nDCG@10", "MRR@10", "Acc@1", "Acc@5", "Acc@10", "R@100", "MAP@100"]:
    print(f"  {col}: {tabla[col].idxmax()} ({tabla[col].max():.4f})")
tabla.to_csv(f"{OUT}/overall_results.csv")
print(f"\nSaved to {OUT}/overall_results.csv. Embeddings are stored in {CACHE} for the other notebooks.")


        nDCG@10  MRR@10   Acc@1   Acc@5  Acc@10   R@100  MAP@100  N_top1
BM25     0.7672  0.7338  0.6568  0.8326  0.8736  0.9465   0.7359    3636
BGE-M3   0.7742  0.7413  0.6671  0.8398  0.8786  0.9486   0.7436    3693
E5       0.7527  0.7200  0.6481  0.8165  0.8573  0.9305   0.7222    3588
GEMMA    0.7815  0.7507  0.6810  0.8409  0.8790  0.9503   0.7532    3770

Best system per metric:
  nDCG@10: GEMMA (0.7815)
  MRR@10: GEMMA (0.7507)
  Acc@1: GEMMA (0.6810)
  Acc@5: GEMMA (0.8409)
  Acc@10: GEMMA (0.8790)
  R@100: GEMMA (0.9503)
  MAP@100: GEMMA (0.7532)

Saved to /content/results/overall_results.csv. Embeddings are stored in /content/cache_emb for the other notebooks.


## Notes

- Each section is independent: the final table uses whatever is stored in `RESULTADOS` when it is run.
- The embeddings saved in `cache_emb/` are reused by the other notebooks, so the models do not need to be run again.
- All models are evaluated zero-shot, with their default settings and no fine-tuning.
